# Paper 2 — Powered Steering Run + Validity Control

**Runtime: GPU, hours. Resumable — interrupt freely.**

Run the stages in order by editing `STAGES` in the config cell and re-running the final cell:

| STAGES | What it does |
|---|---|
| `[0]` | Preflight — verify inputs, print work plan and runtime estimate |
| `[1]` | Generate steered rollouts at REVISION positions |
| `[2]` | Generate steered rollouts at matched FILLER positions |
| `[3]` | Analysis — validity control, UNKNOWN sensitivity, mechanism comparison |

Stage 3 section 2 is the retired scope AUC and is skipped unless you set `SCOPE_ANALYSIS_ENABLED = True`.

Prerequisites on Drive: `counterfactual_pilot_events.json`, `filler_control_pilot_events.json`, and the extracted steering directions.


In [ ]:
# ---- Google Drive -----------------------------------------------------------
from google.colab import drive
drive.mount("/content/gdrive")

In [ ]:
# ---- scope_flag (inlined; no file to place) ---------------------------------
# The analysis scripts below contain `import scope_flag` and gate their retired
# local/global SCOPE sections on it. Rather than editing that logic -- which
# would risk changing behaviour -- this cell installs a real module object of
# that name into sys.modules, so the imports below resolve to it unchanged.
#
# SET THIS ONE TOGGLE:
SCOPE_ANALYSIS_ENABLED = False   # False = paper path (default). True = also
                                 # run the retired scope analysis.
#
# NOTE this governs scope ANALYSIS AND REPORTING only, never sample
# construction. The event sets were drawn stratified on the human
# local/global labels and no flag can change that. See SCOPE_ISOLATION.md §3.

import sys, types

_sf = types.ModuleType("scope_flag")
_sf.SCOPE_ANALYSIS_ENABLED = SCOPE_ANALYSIS_ENABLED
_sf.SCOPE_BANNER_SHOWN = False

def _scope_banner():
    if _sf.SCOPE_BANNER_SHOWN:
        return _sf.SCOPE_ANALYSIS_ENABLED
    _sf.SCOPE_BANNER_SHOWN = True
    if _sf.SCOPE_ANALYSIS_ENABLED:
        print("\n[scope_flag] SCOPE_ANALYSIS_ENABLED=True — exploratory scope "
              "sections WILL run.")
        print("[scope_flag] These are retired/exploratory and are not reported "
              "in the paper.")
    else:
        print("\n[scope_flag] SCOPE_ANALYSIS_ENABLED=False — scope sections "
              "skipped (paper path).")
        print("[scope_flag] Set SCOPE_ANALYSIS_ENABLED=True above to run them.")
    return _sf.SCOPE_ANALYSIS_ENABLED

def _skip_note(section_name):
    print(f"\n[scope_flag] SKIPPED (retired scope analysis): {section_name}")
    print("[scope_flag]   re-enable by setting SCOPE_ANALYSIS_ENABLED = True above")

_sf.scope_banner = _scope_banner
_sf.skip_note = _skip_note
sys.modules["scope_flag"] = _sf
print(f"scope_flag installed. SCOPE_ANALYSIS_ENABLED = {SCOPE_ANALYSIS_ENABLED}")

## Steering run

Edit `STAGES` in the config cell below, then run to the end.


In [ ]:
# =============================================================================
# STEERING: POWERED RUN + VALIDITY CONTROL  (all stages, one file)
#
# WHY
# The steering pilot (n=15, AUC 0.411) has the same unresolved validity problem
# that the filler control exposed for lexical suppression: it measures
# NECESSITY, and necessity was not demonstrated to be revision-specific
# (revision vs filler p=0.127). Steering is a different suppression MECHANISM
# but feeds the same OUTCOME MEASURE. Scaling it without a control would give a
# precise estimate of a quantity that may not be measuring revisions.
#
# So this runs the control FIRST-CLASS alongside the scale-up.
#
# DESIGN: reuses the EXACT event sets from the lexical runs
#   revision arm -> counterfactual_pilot_events.json   (~81 events)
#   filler arm   -> filler_control_pilot_events.json   (~80 events, matched
#                   1:1 on model and marker word)
# Using identical events across mechanisms buys three results from one job:
#   1. VALIDITY  -- steering-revision vs steering-filler necessity   [KEPT]
#   2. POWERED SCOPE -- steering necessity vs human labels at n~81   [RETIRED:
#      gated behind SCOPE_ANALYSIS_ENABLED in scope_flag.py; OFF by default]
#   3. MECHANISM COMPARISON -- lexical vs steering on the SAME events, which is
#      otherwise unobtainable and speaks directly to whether the earlier nulls
#      were mechanism-specific                                        [KEPT]
#
# SCOPE ISOLATION: sections 1, 1b and 3 are the paper's kept results and run
# unconditionally. Section 2 is the retired local/global scope analysis and is
# skipped unless SCOPE_ANALYSIS_ENABLED=1. NOTE that this does not make the
# script scope-free: the revision arm's event set was drawn stratified on the
# human local/global labels (see select_steering_pilot_events.py and
# SCOPE_ISOLATION.md section 3). That is a sample-construction fact, not an
# analysis step, and is disclosed rather than switched off.
#
# The n=15 steering pilot events are NOT reused: they came from a different
# draw (seed 9001) and would not align with the lexical sets.
#
# -----------------------------------------------------------------------------
# HOW TO RUN — set STAGES, then run.
#   [0] Preflight: verify inputs, print the work plan and a runtime estimate.
#       Costs nothing. RUN THIS FIRST.
#   [1] Generate steering rollouts, REVISION arm.   GPU, resumable.
#   [2] Generate steering rollouts, FILLER arm.     GPU, resumable.
#   [3] Analyse: validity control + powered AUC + mechanism comparison. CPU.
#
# Stages 1 and 2 are independently resumable; interrupt freely.
# =============================================================================

STAGES = [0]          # <-- edit this



from pathlib import Path
from collections import defaultdict, Counter
import json, re, time

import numpy as np
from scipy.stats import mannwhitneyu
from sklearn.metrics import roc_auc_score

In [ ]:
import scope_flag   # SCOPE_ANALYSIS_ENABLED gate; see SCOPE_ISOLATION.md

GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
EXPORT_DIR  = GDRIVE_BASE / "hand_labeling_export"

REVISION_EVENTS   = EXPORT_DIR / "counterfactual_pilot_events.json"
FILLER_EVENTS     = EXPORT_DIR / "filler_control_pilot_events.json"
DIRECTIONS_PATH   = EXPORT_DIR / "steering_directions.json"

REV_RESULTS       = EXPORT_DIR / "steering_revision_rollouts_powered.json"

In [ ]:
FLR_RESULTS       = EXPORT_DIR / "steering_filler_rollouts_powered.json"
ANALYSIS_OUT      = EXPORT_DIR / "steering_powered_analysis.json"

# lexical results, for the mechanism comparison in stage 3 (optional)
LEXICAL_REV       = EXPORT_DIR / "counterfactual_rollout_results.json"
LEXICAL_FLR       = EXPORT_DIR / "filler_control_rollout_results.json"

N_ROLLOUTS             = 8       # matches both lexical arms
ROLLOUT_MAX_NEW_TOKENS = 2048    # matches both lexical arms
SAMPLE_TEMPERATURE     = 0.6

In [ ]:
SAMPLE_TOP_P           = 0.95

# From the pilot calibration. Relative to each model's raw_direction_norm (the
# NATURAL revision-vs-filler signal magnitude), not the ambient activation norm.
# Reused rather than recalibrated: the powered event sets are drawn from the
# same population, so the calibration should transfer. If stage 3 shows
# coherence problems (high UNKNOWN extraction rates), recalibrate before
# trusting the numbers.
CHOSEN_ALPHA_MULTIPLIER = {
    "DeepSeek-R1-Distill-Llama-8B":  1.5,
    "DeepSeek-R1-Distill-Qwen-1.5B": 3.0,
    "DeepSeek-R1-Distill-Qwen-7B":   1.5,
    "DeepSeek-R1-Distill-Qwen-14B":  5.0,
}

# Same list the lexical arms used, for leak-filtering parity in stage 3.
# NOTE: steering does NOT ban these during generation -- it suppresses a
# direction. They are used only to filter post-hoc, so both mechanisms are
# filtered identically and the comparison is not confounded by that.
EXPLICITLY_BANNED = [
    "wait", "actually", "let me reconsider", "that's wrong", "that is wrong",
    "i made an error", "let me try again", "this doesn't work",
    "this does not work", "i need to reconsider", "hmm", "that's not right",
    "that is not right", "let me rethink", "on second thought",
    "let me go back", "wait, let me", "actually, let me", "let me recheck",
    "let me re-check", "alternatively", "i made a mistake",
    "i think i made a mistake", "let me double check", "let me double-check",
    "hold on",
]


# ============================================================================
# shared
# ============================================================================
def is_leaked(text):
    t = text.lower()
    return any(p in t for p in EXPLICITLY_BANNED)


def extract_answer(text: str) -> str:
    """Verbatim from Task_1_Data_Pipeline.ipynb."""
    boxed = re.findall(r'\\boxed\{([^}]+)\}', text)
    if boxed:
        return boxed[-1].strip()
    m = re.search(r'(?:answer is|therefore)[:\s]*([0-9A-Za-z\-\.\/]+)',
                  text, re.IGNORECASE)
    return m.group(1).strip() if m else "UNKNOWN"


def build_prompt(problem: dict, tokenizer, model_name: str) -> str:
    """Verbatim from Task_1_Data_Pipeline.ipynb."""
    is_reasoning = any(tag in model_name for tag in ["R1", "QwQ"])
    if problem.get("difficulty") == "hard":
        instruction = ("Solve the following competition mathematics problem carefully. "
                        "Show your reasoning step by step and verify your answer.")
    else:
        instruction = ("Solve the following mathematics competition problem. "
                        "Show your working step by step.")
    if is_reasoning:
        messages = [{"role": "user", "content": f"{instruction}\n\nProblem: {problem['problem']}"}]
    else:
        messages = [
            {"role": "system", "content": "You are an expert mathematician. Solve problems with "
                                          "clear step-by-step reasoning."},
            {"role": "user", "content": f"{instruction}\n\nProblem: {problem['problem']}"}
        ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def necessity(rollout_correct):
    if not rollout_correct:
        return None
    return 1.0 - sum(rollout_correct) / len(rollout_correct)

In [ ]:
def load_json(p, default=None):
    return json.load(open(p)) if p.exists() else default


# ============================================================================
# STAGE 0 — preflight
# ============================================================================
def stage0_preflight():
    print(f"\n{'#'*70}\n# STAGE 0 — PREFLIGHT\n{'#'*70}\n")
    ok = True
    for label, p in [("revision events", REVISION_EVENTS),
                     ("filler events", FILLER_EVENTS),
                     ("steering directions", DIRECTIONS_PATH)]:
        exists = p.exists()
        ok &= exists
        print(f"  {'OK ' if exists else 'MISSING'}  {label:<22} {p.name}")
    if not ok:
        print("\nMissing inputs -- cannot proceed. Check paths above.")
        return

    rev = load_json(REVISION_EVENTS, [])
    flr = load_json(FILLER_EVENTS, [])
    dirs = load_json(DIRECTIONS_PATH, {})

    print(f"\n  revision events : {len(rev)}")
    print(f"  filler events   : {len(flr)}")
    print(f"  directions for  : {len(dirs)} models")

    missing_alpha = [m for m in dirs if m not in CHOSEN_ALPHA_MULTIPLIER]
    if missing_alpha:
        print(f"\n  WARNING: no alpha for {missing_alpha}")

    print(f"\n  per-model breakdown:")
    cr, cf = Counter(e["model"] for e in rev), Counter(e["model"] for e in flr)
    for m in sorted(set(cr) | set(cf)):
        a = CHOSEN_ALPHA_MULTIPLIER.get(m)
        d = dirs.get(m, {})
        print(f"    {m:<32} rev {cr.get(m,0):>3}  fil {cf.get(m,0):>3}  "
              f"alpha_mult {a}  layer {d.get('layer','?')}")

    done_rev = {(r["problem_id"], r["model"]) for r in load_json(REV_RESULTS, [])}
    done_flr = {(r["problem_id"], r["model"]) for r in load_json(FLR_RESULTS, [])}
    todo_rev = len(rev) - sum(1 for e in rev if (e["problem_id"], e["model"]) in done_rev)
    todo_flr = len(flr) - sum(1 for e in flr if (e["problem_id"], e["model"]) in done_flr)

    total_gen = (todo_rev + todo_flr) * N_ROLLOUTS
    # lexical filler run: ~80 events in ~10h -> ~450s per event of 8 rollouts
    est_h = (todo_rev + todo_flr) * 450 / 3600
    print(f"\n  remaining work: {todo_rev} revision + {todo_flr} filler events")
    print(f"                  = {total_gen} generations")
    print(f"  rough estimate: ~{est_h:.0f} h  (based on the lexical filler run's pace;")
    print(f"                  Qwen-14B is materially slower than the others)")
    print(f"\n  Both generation stages are resumable -- interrupt freely.")


# ============================================================================
# STAGES 1 & 2 — steering generation
# ============================================================================
def _generate(events_path, results_path, arm_label):
    print(f"\n{'#'*70}\n# STEERING GENERATION — {arm_label} ARM\n{'#'*70}\n")
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    events = load_json(events_path)
    dirs = load_json(DIRECTIONS_PATH)
    assert events is not None and dirs is not None, "missing inputs -- run stage 0"
    print(f"Loaded {len(events)} {arm_label.lower()} events.")

    missing = [m for m in {e["model"] for e in events} if m not in CHOSEN_ALPHA_MULTIPLIER]
    if missing:
        raise ValueError(f"No alpha for {missing}. Fill CHOSEN_ALPHA_MULTIPLIER.")

    def get_layers(model):
        assert hasattr(model, "model") and hasattr(model.model, "layers"), (
            "Expected model.model.layers (standard Llama/Qwen2 layout).")
        return model.model.layers

    def make_hook(direction, alpha):
        def hook(module, inputs, output):
            if isinstance(output, tuple):
                return (output[0] - alpha * direction,) + output[1:]
            return output - alpha * direction
        return hook

    @torch.no_grad()
    def gen(model, tokenizer, prefix, layer_idx, direction, alpha, seed):
        torch.manual_seed(seed)
        inputs = tokenizer(prefix, return_tensors="pt", truncation=False).to(model.device)
        handle = None
        if alpha != 0.0:
            handle = get_layers(model)[layer_idx].register_forward_hook(
                make_hook(direction, alpha))
        try:
            out = model.generate(
                **inputs, max_new_tokens=ROLLOUT_MAX_NEW_TOKENS, do_sample=True,
                temperature=SAMPLE_TEMPERATURE, top_p=SAMPLE_TOP_P,
                pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
        finally:
            if handle is not None:
                handle.remove()   # always, even on exception
        return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                                skip_special_tokens=False)

    all_results = load_json(results_path, [])
    done = {(r["problem_id"], r["model"]) for r in all_results}
    if done:
        print(f"Resuming: {len(done)} events already complete.")

    by_model = defaultdict(list)
    for e in events:
        by_model[e["model"]].append(e)

    for model_short, evs in by_model.items():
        remaining = [e for e in evs if (e["problem_id"], model_short) not in done]
        if not remaining:
            print(f"\n{model_short}: complete -- skipping model load.")
            continue

        info = dirs[model_short]
        layer_idx = info["layer"]
        alpha = CHOSEN_ALPHA_MULTIPLIER[model_short] * info["raw_direction_norm"]
        model_full = f"deepseek-ai/{model_short}"
        print(f"\n{'='*62}\n{model_full}  ({len(remaining)} remaining)")
        print(f"layer {layer_idx}  alpha_mult {CHOSEN_ALPHA_MULTIPLIER[model_short]}  "
              f"alpha {alpha:.3f}\n{'='*62}")

        tokenizer = AutoTokenizer.from_pretrained(model_full, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            model_full, torch_dtype=torch.float16, device_map="auto",
            trust_remote_code=True)
        model.eval()
        direction = torch.tensor(info["unit_direction"], dtype=torch.float16,
                                  device=model.device)

        for ev in remaining:
            print(f"\n  {ev['problem_id']}  ({arm_label.lower()})")
            prompt = build_prompt(
                {"problem": ev["problem"], "difficulty": ev["difficulty"]},
                tokenizer, model_full)
            prefix = prompt + ev["prefix_response_text"]
            correct, texts = [], []
            t0 = time.time()
            for r in range(N_ROLLOUTS):
                txt = gen(model, tokenizer, prefix, layer_idx, direction, alpha, 4000 + r)
                ans = extract_answer(txt)
                okk = ans.strip() == ev["answer_gt"].strip()
                correct.append(okk); texts.append(txt)
                print(f"    {r+1}/{N_ROLLOUTS}: ans={ans!r} gt={ev['answer_gt']!r} -> {okk}")
            print(f"  necessity={necessity(correct):.3f}  ({time.time()-t0:.0f}s)")

            all_results.append({
                "problem_id": ev["problem_id"], "model": model_short,
                "human_label": ev.get("human_label"),
                "rollout_correct": correct, "rollout_texts": texts,
                "alpha": alpha, "layer": layer_idx, "arm": arm_label.lower(),
            })
            with open(results_path, "w") as f:
                json.dump(all_results, f, indent=2)

        del model
        torch.cuda.empty_cache()

    print(f"\nWrote {len(all_results)} results -> {results_path}")


def stage1_revision():
    _generate(REVISION_EVENTS, REV_RESULTS, "REVISION")


def stage2_filler():
    _generate(FILLER_EVENTS, FLR_RESULTS, "FILLER")


# ============================================================================
# STAGE 3 — analysis
# ============================================================================
def _scores(path, label, apply_leak_filter):
    """
    apply_leak_filter MUST be False for steering arms.

    Leak-filtering exists to remove rollouts that escaped an ACTIVE token ban.
    Steering bans nothing -- it suppresses a direction in activation space --
    so a steered rollout containing "wait" has not leaked past anything. It is
    just ordinary output.

    Applying the lexical ban list to steering output anyway destroyed the first
    run: it removed 556/482 rollouts and 48/39 whole events (53% of the run),
    versus 46/48 rollouts and 0 events for the lexical arms. Worse, the
    survivors are exactly the rollouts where the model happened not to use
    common discourse words -- a strongly non-random subset, biased in an
    unknown direction. "Filtered identically" is not the same as "filtered
    fairly" when the filter means something different for each mechanism.
    """
    results = load_json(path, [])
    out, lost_ev, lost_ro, n_unknown, n_total_ro = [], 0, 0, 0, 0
    for r in results:
        texts = r.get("rollout_texts", [])
        if apply_leak_filter and texts:
            clean = [c for c, t in zip(r["rollout_correct"], texts) if not is_leaked(t)]
            lost_ro += len(r["rollout_correct"]) - len(clean)
        else:
            clean = r["rollout_correct"]
        # extraction-quality diagnostic: how many failures are UNKNOWN/malformed
        # rather than a genuinely different answer? These inflate necessity.
        for t in texts:
            n_total_ro += 1
            if extract_answer(t) == "UNKNOWN":
                n_unknown += 1
        if not clean:
            lost_ev += 1
            continue
        out.append({"model": r["model"], "problem_id": r["problem_id"],
                    "human_label": r.get("human_label"),
                    "necessity": necessity(clean), "n_clean": len(clean),
                    # retained so the UNKNOWN sensitivity analysis can recompute
                    # necessity under alternative treatments without re-running
                    "rollout_correct": list(clean),
                    "rollout_unknown": [extract_answer(t) == "UNKNOWN" for t in texts]
                                        if texts else []})
    filt = "leak-filtered" if apply_leak_filter else "UNFILTERED (correct for steering)"
    print(f"  {label}: {len(out)} events [{filt}]")
    if apply_leak_filter:
        print(f"      {lost_ro} rollouts dropped, {lost_ev} events lost entirely")
    if n_total_ro:
        print(f"      extraction: {n_unknown}/{n_total_ro} rollouts "
              f"({n_unknown/n_total_ro:.0%}) gave UNKNOWN -- these count as")
        print(f"      incorrect and therefore inflate necessity in BOTH arms")
    return out


def _describe(scores, label):
    v = np.array([s["necessity"] for s in scores])
    if len(v) == 0:
        print(f"\n  {label}: no data")
        return v
    nz = int((v == 0).sum())
    print(f"\n  {label}  (n={len(v)})")
    print(f"    mean {v.mean():.3f}  median {np.median(v):.3f}  sd {v.std():.3f}")
    print(f"    exactly zero {nz}/{len(v)} ({nz/len(v):.0%})   "
          f"fully necessary {int((v==1.0).sum())}/{len(v)}")
    return v

### Run

Edit `STAGES` above, run that cell, then run the cell below. Repeat for each stage: `[0]` → `[1]` → `[2]` → `[3]`.


In [ ]:
# ---- WHICH STAGES TO RUN  (edit this, then run this cell and the next) -----
# This overrides the STAGES value set in the config cell above, so you can
# change it between runs without scrolling back up.
#
#   [0] preflight (fast)   [1] revision rollouts (GPU, hours)
#   [2] filler rollouts (GPU, hours)   [3] analysis (fast)
#
# Stages 1 and 2 are resumable -- interrupt freely and re-run.

STAGES = [0]

print(f"STAGES = {STAGES}")

In [ ]:
def _mw(a, b):
    """Reports BOTH median and mean differences.

    Necessity is quantized to 1/N_ROLLOUTS (eighths), and both arms pile up at
    the low end, so the median is frequently identical between arms even when
    the rank test detects a clear difference. Relying on median_diff alone
    caused a real misread: p=0.033 with median_diff=0.000 was reported as
    'validity not established'. The mean and the rank statistic are the
    informative summaries here.
    """
    if len(a) < 3 or len(b) < 3:
        return None
    a, b = np.asarray(a, float), np.asarray(b, float)
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    auc = u / (len(a) * len(b))
    rng = np.random.RandomState(0)
    med_d, mean_d = [], []
    for _ in range(5000):
        sa, sb = rng.choice(a, len(a), True), rng.choice(b, len(b), True)
        med_d.append(np.median(sa) - np.median(sb))
        mean_d.append(sa.mean() - sb.mean())
    return {"p": float(p), "auc": float(auc),
            "median_diff": float(np.median(a) - np.median(b)),
            "median_ci95": [float(x) for x in np.percentile(med_d, [2.5, 97.5])],
            "mean_diff": float(a.mean() - b.mean()),
            "mean_ci95": [float(x) for x in np.percentile(mean_d, [2.5, 97.5])],
            "n_a": len(a), "n_b": len(b)}


def stage3_analyse():
    print(f"\n{'#'*70}\n# STAGE 3 — ANALYSIS\n{'#'*70}\n")
    rev = _scores(REV_RESULTS, "steering revision", apply_leak_filter=False)
    flr = _scores(FLR_RESULTS, "steering filler  ", apply_leak_filter=False)
    results = {}

    # ---- 1. VALIDITY: is steering-necessity revision-specific? -------------
    print(f"\n{'='*70}\n1. VALIDITY CONTROL — revision vs matched filler (steering)\n{'='*70}")
    v_rev = _describe(rev, "REVISION positions")
    v_flr = _describe(flr, "FILLER positions (matched)")
    val = _mw(v_rev, v_flr) if len(v_rev) and len(v_flr) else None
    if val:
        print(f"\n  Mann-Whitney p = {val['p']:.4f}   P(rev > fil) = {val['auc']:.3f}")
        print(f"  mean   difference {val['mean_diff']:+.4f}  "
              f"95% CI [{val['mean_ci95'][0]:+.4f}, {val['mean_ci95'][1]:+.4f}]  <- informative")
        print(f"  median difference {val['median_diff']:+.4f}  "
              f"95% CI [{val['median_ci95'][0]:+.4f}, {val['median_ci95'][1]:+.4f}]"
              f"  <- coarse: necessity is quantized to 1/{N_ROLLOUTS}")
        results["validity"] = val

    # ---- 1b. UNKNOWN SENSITIVITY -------------------------------------------
    # UNKNOWN extractions score as incorrect, so they INFLATE necessity. If the
    # arms have different UNKNOWN rates, that asymmetry could manufacture the
    # validity result rather than reflect a real difference. This recomputes
    # the comparison three ways. The conclusion must hold under all three, or
    # it is an extraction artifact rather than a finding.
    def rate_unknown(scores):
        tot = sum(len(s["rollout_unknown"]) for s in scores)
        unk = sum(sum(s["rollout_unknown"]) for s in scores)
        return unk / tot if tot else 0.0

    def necessity_variant(scores, mode):
        """mode: 'as_incorrect' (default), 'dropped', 'as_correct'."""
        out = []
        for s in scores:
            cor, unk = s["rollout_correct"], s["rollout_unknown"]
            if not unk or len(unk) != len(cor):
                out.append(s["necessity"]); continue
            if mode == "as_incorrect":
                vals = cor
            elif mode == "dropped":
                vals = [c for c, u in zip(cor, unk) if not u]
            elif mode == "as_correct":
                vals = [True if u else c for c, u in zip(cor, unk)]
            if vals:
                out.append(1.0 - sum(vals) / len(vals))
        return np.array(out)

    print(f"\n{'-'*70}\n1b. UNKNOWN SENSITIVITY  (rev {rate_unknown(rev):.0%} vs "
          f"fil {rate_unknown(flr):.0%} UNKNOWN)\n{'-'*70}")
    print("  UNKNOWN is NOT missing-at-random -- it correlates with degraded")
    print("  generation, which suppression causes. So neither 'as incorrect' nor")
    print("  'dropped' is obviously right; the two bracket the truth.")
    sens = {}
    for mode, desc in [("as_incorrect", "as INCORRECT (default; inflates necessity)"),
                       ("dropped",      "DROPPED (removes worst-affected rollouts)"),
                       ("as_correct",   "as CORRECT (extreme lower bound)")]:
        a, b = necessity_variant(rev, mode), necessity_variant(flr, mode)
        r = _mw(a, b)
        if r:
            flag = "*" if r["p"] < 0.05 else " "
            print(f"  {desc:<44} rev {a.mean():.3f} fil {b.mean():.3f}  "
                  f"diff {r['mean_diff']:+.3f}  p={r['p']:.4f}{flag}")
            sens[mode] = r
    results["unknown_sensitivity"] = sens
    # The three treatments are NOT equally plausible and must not be weighted
    # equally:
    #   as_incorrect - the default. Defensible: an unparseable output did not
    #                  produce the right answer, so scoring it wrong is fair.
    #   dropped      - defensible: treat unparseable as missing data.
    #   as_correct   - an EXTREME STRESS TEST, not a plausible alternative. It
    #                  asserts the model got the right answer every time the
    #                  parser failed. Failing this is expected and is not
    #                  evidence against the finding.
    # Robustness is therefore judged on the two DEFENSIBLE treatments, with
    # as_correct reported for transparency as a worst case.
    PLAUSIBLE = ["as_incorrect", "dropped"]
    n_plaus = sum(1 for m in PLAUSIBLE
                  if m in sens and sens[m]["p"] < 0.05 and sens[m]["mean_diff"] > 0)
    all_positive = all(r["mean_diff"] > 0 for r in sens.values())
    stress = sens.get("as_correct")

    print(f"\n  -> significant and positive in {n_plaus}/{len(PLAUSIBLE)} DEFENSIBLE "
          f"treatments (as_incorrect, dropped)")
    if stress:
        s_sig = stress["p"] < 0.05 and stress["mean_diff"] > 0
        print(f"     stress test (as_correct): diff {stress['mean_diff']:+.3f}, "
              f"p={stress['p']:.4f} -> {'also holds' if s_sig else 'does not hold (expected)'}")
    print(f"     direction positive in all three: {all_positive}")

    if n_plaus == len(PLAUSIBLE):
        print("\n     ROBUST: holds under both defensible treatments of unparseable")
        print("     output. Report the sensitivity table, but the validity finding")
        print("     stands. Note it fails only the deliberately implausible bound")
        print("     that assumes every unparseable output was correct.")
        robust_verdict = "robust"
    elif n_plaus == 1:
        print("\n     PARTIAL: holds under one defensible treatment but not the other.")
        print("     Extraction handling materially affects the conclusion. Report all")
        print("     three and qualify the claim.")
        robust_verdict = "partial"
    else:
        print("\n     FRAGILE: fails under both defensible treatments. The headline")
        print("     result is likely an extraction artifact -- do not report it as")
        print("     a validity finding.")
        robust_verdict = "fragile"
    results["unknown_sensitivity_verdict"] = robust_verdict

    # ---- 2. POWERED SCOPE: necessity vs human labels -----------------------
    # RETIRED / EXPORATORY -- gated behind SCOPE_ANALYSIS_ENABLED. This is the
    # only scope-dependent section in stage 3; sections 1, 1b and 3 are the
    # paper's kept results and run unconditionally. See SCOPE_ISOLATION.md.
    if scope_flag.SCOPE_ANALYSIS_ENABLED:
        scope_flag.scope_banner()
        print(f"\n{'='*70}\n2. POWERED SCOPE TEST — necessity vs human local/global"
              f"\n   [RETIRED EXPLORATORY ANALYSIS — not reported in the paper]\n{'='*70}")
        loc = [s["necessity"] for s in rev if s["human_label"] == "local"]
        glo = [s["necessity"] for s in rev if s["human_label"] == "global"]
        print(f"  n_local={len(loc)}  n_global={len(glo)}")
        if len(loc) >= 3 and len(glo) >= 3:
            auc = roc_auc_score([1]*len(glo) + [0]*len(loc), glo + loc)
            _, p = mannwhitneyu(glo, loc, alternative="two-sided")
            # Hanley-McNeil CI
            q1, q2 = auc/(2-auc), 2*auc**2/(1+auc)
            se = np.sqrt((auc*(1-auc) + (len(glo)-1)*(q1-auc**2)
                          + (len(loc)-1)*(q2-auc**2)) / (len(glo)*len(loc)))
            lo, hi = max(0, auc-1.96*se), min(1, auc+1.96*se)
            print(f"  median necessity -- local {np.median(loc):.3f}  global {np.median(glo):.3f}")
            print(f"  AUC = {auc:.3f}   95% CI [{lo:.3f}, {hi:.3f}]   p = {p:.4f}")
            print(f"  (pilot at n=15 gave AUC 0.411)")
            results["scope"] = {"auc": float(auc), "ci95": [float(lo), float(hi)],
                                "p": float(p), "n_local": len(loc), "n_global": len(glo)}
        else:
            print("  insufficient labelled events in one class")
    else:
        scope_flag.scope_banner()
        scope_flag.skip_note("stage 3 section 2 — necessity vs human local/global AUC")

    # ---- 3. MECHANISM COMPARISON: lexical vs steering, same events ---------
    print(f"\n{'='*70}\n3. MECHANISM COMPARISON — lexical vs steering (same events)\n{'='*70}")
    print("  Reported two ways, because the arms are not natively comparable:")
    print("    (a) lexical AS PUBLISHED (leak-filtered) vs steering unfiltered")
    print("    (b) BOTH unfiltered -- like-for-like, no filter applied anywhere")
    lex_rev_f = _scores(LEXICAL_REV, "lexical revision (filtered)  ",
                        apply_leak_filter=True) if LEXICAL_REV.exists() else []
    lex_rev_u = _scores(LEXICAL_REV, "lexical revision (unfiltered)",
                        apply_leak_filter=False) if LEXICAL_REV.exists() else []

    def compare_mechanisms(lex_scores, tag):
        if not (lex_scores and rev):
            return None
        lk = {(s["model"], s["problem_id"]): s["necessity"] for s in lex_scores}
        sk = {(s["model"], s["problem_id"]): s["necessity"] for s in rev}
        shared = sorted(set(lk) & set(sk))
        print(f"\n  --- {tag} ---")
        print(f"  events under BOTH mechanisms: {len(shared)}")
        if len(shared) < 10:
            print("  too few shared events")
            return None
        a = np.array([lk[k] for k in shared]); b = np.array([sk[k] for k in shared])
        from scipy.stats import spearmanr, wilcoxon
        rho, prho = spearmanr(a, b)
        print(f"  lexical median {np.median(a):.3f}   steering median {np.median(b):.3f}")
        print(f"  Spearman rho = {rho:+.3f}  p = {prho:.4f}")
        pw = None
        try:
            _, pw = wilcoxon(a, b)
            print(f"  Wilcoxon signed-rank (paired) p = {pw:.4f}"
                  f"   <- tests whether LEVELS differ, not whether they correlate")
        except Exception:
            pass
        return {"n_shared": len(shared), "spearman_rho": float(rho),
                "spearman_p": float(prho), "wilcoxon_p": float(pw) if pw else None,
                "median_lexical": float(np.median(a)),
                "median_steering": float(np.median(b))}

    m_a = compare_mechanisms(lex_rev_f, "(a) lexical filtered vs steering unfiltered")
    m_b = compare_mechanisms(lex_rev_u, "(b) both unfiltered — like-for-like")
    if m_a: results["mechanism_lexfiltered"] = m_a
    if m_b: results["mechanism_both_unfiltered"] = m_b

    ref = m_b or m_a
    if ref:
        print(f"\n  Correlated (rho > 0, p < 0.05) -> both mechanisms track the same")
        print(f"  underlying quantity, so the earlier nulls are not mechanism artifacts.")
        print(f"  Note Spearman and Wilcoxon answer different questions: rank agreement")
        print(f"  vs. difference in level. Both can hold at once.")

    # ---- read ---------------------------------------------------------------
    print(f"\n{'#'*70}\n# READ\n{'#'*70}")
    # NB: gate on the RANK TEST and the MEAN, never median_diff. With necessity
    # quantized to eighths and both arms massed at the low end, medians are
    # routinely identical even when the distributions clearly differ. An earlier
    # version required median_diff > 0 and consequently reported a significant
    # result (p=0.033) as "not established".
    passed = val and val["p"] < 0.05 and val["mean_diff"] > 0
    robust = (robust_verdict == "robust")

    if passed and robust:
        print("VALIDITY PASSED, and robust to defensible UNKNOWN treatments.")
        print(f"  Steering-necessity is higher at revisions than at matched fillers")
        print(f"  (p = {val['p']:.4f}, mean diff {val['mean_diff']:+.3f}, "
              f"P(rev>fil) = {val['auc']:.3f}), and the effect persists when")
        print(f"  unparseable outputs are excluded rather than scored incorrect.")
        print("\n  This measurement IS revision-specific -- unlike lexical suppression,")
        print("  which failed the same control on the uncorrected parser. NOTE: with the")
        print("  corrected parser and repaired ground truth, BOTH mechanisms pass; see")
        print("  Paper2_CORRECTED_Both_Mechanisms_Pass.md. Do not cite p=0.127 as a")
        print("  lexical failure without that correction.")
        if scope_flag.SCOPE_ANALYSIS_ENABLED:
            print("\n  CONSEQUENCE for the retired scope analysis: the scope null above")
            print("  can no longer be dismissed as downstream of a broken instrument.")
            print("  It is a well-powered null on a measurement demonstrated to respond")
            print("  to revisions. Retained for the record only -- the scope claim is")
            print("  not made in the paper.")
    elif passed and not robust:
        print("VALIDITY PASSED on the headline test, but NOT robust to UNKNOWN handling.")
        print(f"  Headline: p = {val['p']:.4f}, mean diff {val['mean_diff']:+.3f}.")
        print("  The result changes under alternative UNKNOWN treatments (see 1b), and")
        print("  the arms have different UNKNOWN rates, so extraction asymmetry is a live")
        print("  explanation. Do NOT report validity as established without reporting the")
        print("  sensitivity analysis alongside it.")
    elif val:
        print(f"VALIDITY NOT ESTABLISHED (p = {val['p']:.4f}, mean diff "
              f"{val['mean_diff']:+.3f}).")
        print("  Steering-necessity is not distinguishable from matched fillers. Both")
        print("  suppression mechanisms then fail the same control, which is a cleaner")
        print("  claim than either alone: necessity as operationalised is not")
        print("  revision-specific regardless of how suppression is implemented.")
    print("\nCheck CI widths before calling any null conclusive.")

    json.dump(results, open(ANALYSIS_OUT, "w"), indent=2)
    print(f"\nWrote {ANALYSIS_OUT}")


# ============================================================================
if __name__ == "__main__":
    if 0 in STAGES: stage0_preflight()
    if 1 in STAGES: stage1_revision()
    if 2 in STAGES: stage2_filler()
    if 3 in STAGES: stage3_analyse()
    print(f"\nDone. Stages: {STAGES}")